In [1]:
# function takes original file relative path, new fline name that is going to encod, K value for number of bits to use for encoding 
def makeEncodedFile(ogFileName, newEncodedFileName, K):

  #  --* GET FILE VALUES - LIST OF INTS *-- 
  
  # import wave to get byte object data from wav file 
  import wave
  ogSound = wave.open(ogFileName, mode='rb')
  # Extract all the bytes from the file 
  ogSoundBytes = ogSound.readframes(-1)
  # store parameters 
  ogSoundParameters = ogSound.getparams()
  ogSound.close() # close the resource 
  # turn byte object to the list of ints 
  ogSoundIntList = list(ogSoundBytes) 


  # --* ENCODE INT LIST FROM FILE TO BINARY STRING *-- 
  # with 2 inner function
  # encode - take int value and return encoded binary 
  # encodeList - take list of ints and return binary string 

  # encode - take int value and return encoded binary 
  def encode(S, K):
    # S - input sample
    # K - number of bits 
    # M - calculated M = 2^K
    M = pow(2, K)
    # q - quotient 
    q = int(S/M)
    # q is in int, change it to unary 
    unaryq = ''
    for i in range(q):
      unaryq = unaryq + '1'
    unaryq = unaryq + '0'
    # r - remainder
    # r = SmoduloM
    r = S % M
    # r is in int, change it to K size binary
    binarySize = '0' + str(K) + 'b' # this allows change to be dynamic based on K 
    binaryr = format(r, binarySize) # format(r, '0Kb) where k is a number that will result in binary to have 0 in front if nessesery
    result = unaryq + str(binaryr)
    return result
  
  # encodeList
  def encodeList(values, K):
    result = ''
    for value in values:
      result = result + encode(value, K)
    return result 

  # use those inner functions make binary string 
  encodedBinaryStr = encodeList(ogSoundIntList, K)


  # --* Binary string to bytes object *--
  # concept is to store this string in byte format, to minimize file size 
  # to accomplish that without loosing any value during decoding I came up with idea to 
  # add 0s before to insure that whole encoded string is devidible by 8 
  # but in case where first encoded value is 0 or the number that would make unary to start with 0 not 1 
  # I decided to also add hard encoded value that starts with 1 in this case max value of 255 
  
  # binaryStrToBytes - takes strValue and K returns byte object with 0s and 255 addded as prefix 
  def binaryStrToBytes(strValue,K):
    # first add 255 at the begining 
    result = encode(255,K)
    result = result + strValue
    # add zeroes to make stirng devidable by 8 
    numOfZeroes = ''
    for i in range (len(strValue) % 8):
      numOfZeroes = numOfZeroes + '0'
    result = numOfZeroes + result
    # next step is changing binary string to bytes object, I was thinking to make loop and change every 8 characters to byte,
    # but I looked online for to see if there is some more efficient built in python function I found: 
    # I DID NOT WRITE CODE BELOW COPIED FROM : Title: Convert binary string to bytearray in Python 3
    # https://stackoverflow.com/questions/32675679/convert-binary-string-to-bytearray-in-python-3 Author: PM 2Ring , Sep 20, 2015, accessed: 20 Aug 2024
    # (I changed variable 's' to  'result' to work with my code) 
    return int(result, 2).to_bytes(len(result) // 8, byteorder='big')
    # END OF THE CODE I DID NOT WRITE 

  # 
  encodedBytes = binaryStrToBytes(encodedBinaryStr, K)


  # --* Create new file and write data to it *-- 

  newSound = wave.open(newEncodedFileName, mode='wb')
  # from list of ints to bytearray to byte object 
  newSound.setparams(ogSoundParameters)
  newSound.writeframes(encodedBytes)
  newSound.close()
  return 

In [6]:
# encode Sound1.wav K = 4
makeEncodedFile("Exercise2_Files/Sound1.wav", "Exercise2_Files/Sound1encodedK4.ex2", 4)

# encode Sound1.wav K = 2
makeEncodedFile("Exercise2_Files/Sound1.wav", "Exercise2_Files/Sound1encodedK2.ex2", 2)

# encode Sound1.wav K = 4
makeEncodedFile("Exercise2_Files/Sound1.wav", "Exercise2_Files/Sound2encodedK4.ex2", 4)

# encode Sound2.wav K = 2
makeEncodedFile("Exercise2_Files/Sound2.wav", "Exercise2_Files/Sound2encodedK2.ex2", 2)

In [3]:
# function takes encoded file relative path, file name for new decoded file, K value for number of bits to use for decoding
def makeDecodedFile(encodedFileName, newDecodedFileName, K):

  #  --* GET FILE VALUES - LIST OF INTS *-- 

  # import wave to get byte object data from wav file 
  import wave
  # take encoded file 
  encodedSound = wave.open(encodedFileName, mode='rb')
  # Extract all the bytes 
  encodedSoundBytes = encodedSound.readframes(-1) # data is in byte object type 
  encodedSoundParameters = encodedSound.getparams()
  encodedSound.close()
  # byte object into list of ints 
  encodedSoundIntList = list(encodedSoundBytes) 


  # --* Int list into binary string *--
  
  encodedBinaryList = []
  for i in range (len(encodedSoundIntList)):
      encodedBinaryList.append(format(encodedSoundIntList[i], '08b')) 
  encodedBinaryStr = ''.join(encodedBinaryList)


  # --* Decode binary str to list of ints *--
  # with 2 inner functions 
  # decode - takes one binary input and decodes it into int 
  # decodeStr - takes mulitple encoded binary values and returns decoded list of ints 

  # decoding function 
  def decode(E, K):
    # E - encoded input in binary 
    # K - number of bits 
    # M - is 2^K
    M = pow(2,K)

    # q - get unary part of the input first 
    q = 0
    for i in E: # loop over every character in E
      if i == '1': # count repetitions of 1 
        q = q + 1
      else: # breake when come across first 0 which means the end unary number 
        break
    
    # r - is last K bits of encoded inuput 
    rBinary = E[q+1:] # since q is number of '1's then 0, after that the rest of E is r in binary 
    r = int(rBinary, 2)
    # Result is q x M + r
    result = q * M + r
    return result

  def decodeStr(encodedBinaryStr,K):
    # get rid of '0'
    encodedBinaryStr = encodedBinaryStr.lstrip('0')
    decodedIntList = []
    # go over every charater of binaryStr, skip the reminders 
    # get index of begining and end for encoded chunk, unary + K bits 
    # decode each chunk and add it to decodedIntList 
    startIndex = 0
    endIndex = 0
    skip = 0
    newChunk = True # to reset begining index for new chunk 
    for i, char in enumerate(encodedBinaryStr):
      # skip when looping over reminder of rice encoding 
      if skip > 0:
        skip = skip - 1
        continue

      # set start index 
      if newChunk:
        startIndex = i
        newChunk = False

      if char == '1': # if '1' it means its 1 from unary 
        endIndex = endIndex + 1 
      else:  # if '0' means that unary is done count that 0, and add next K number of characters, skip iterating over next K characters
        endIndex = endIndex + 1 + K
        skip = K
        newChunk = True
        decodedIntList.append(decode(encodedBinaryStr[startIndex:endIndex], K))
    # return without first 255 it was hard coded by encoding function to make sure first unary starts with 1 
    return decodedIntList[1:]

  # decode string into int list 
  decodedIntList = decodeStr(encodedBinaryStr, K)

  # --* Create new file, write decoded data *--
  #create new file 
  decodedSound = wave.open(newDecodedFileName, mode='wb')
  decodedSound.setparams(encodedSoundParameters)
  decodedBytes = bytes(decodedIntList) # bytes
  decodedSound.writeframes(decodedBytes)
  decodedSound.close()


  return




In [7]:
# Decode Sound1encoded.ex2 K = 4
makeDecodedFile( "Exercise2_Files/Sound1encodedK4.ex2", "Exercise2_Files/Sound1decodedK4.wav", 4)

# Decode Sound1encoded.ex2 K = 2
makeDecodedFile( "Exercise2_Files/Sound1encodedK2.ex2", "Exercise2_Files/Sound1decodedK2.wav", 2)

# Decode Sound2encoded.ex2 K = 4 
makeDecodedFile( "Exercise2_Files/Sound2encodedK4.ex2", "Exercise2_Files/Sound2decodedK4.wav", 4)

# Decode Sound2encoded.ex2 K = 2
makeDecodedFile( "Exercise2_Files/Sound2encodedK2.ex2", "Exercise2_Files/Sound2decodedK2.wav", 2)
